# Quantize the student and measure it on CPU

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps git+https://github.com/sourav-ydv/receipt-router.git

In [ ]:
import os
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets/souravydv25"))

## Settings

In [ ]:
import glob
import json
import statistics
import time

DATA_DIR = "/kaggle/input/datasets/souravydv25/receipt-sft"
ADAPTER_DIR = "/kaggle/input/datasets/souravydv25/receipt-adapter"
QUANT = "q4_k_m"
HF_REPO = ""
print(os.listdir(ADAPTER_DIR))

## Merge and export to GGUF

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=False,
    dtype=torch.float16,
)

model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method=QUANT)

In [ ]:
paths = sorted(glob.glob("**/*.gguf", recursive=True))
for path in paths:
    print(path, round(os.path.getsize(path) / 1e9, 2), "GB")

## CPU runtime

In [ ]:
%%capture
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
from llama_cpp import Llama

from receipts.formatting import INSTRUCTION
from receipts.schema import Receipt
from receipts.scoring import score, summarize
from receipts.validators import check, parse_receipt

gguf_path = [p for p in paths if QUANT in p.lower()][0]
llm = Llama(model_path=gguf_path, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)

with open(f"{DATA_DIR}/test.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f if line.strip()]
test = [row for row in test if row["usable"]]
print(len(test), "receipts,", os.cpu_count(), "cpu threads")

## Evaluate on the test split

In [ ]:
def ask(text):
    started = time.time()
    reply = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": INSTRUCTION},
            {"role": "user", "content": text},
        ],
        temperature=0,
        max_tokens=800,
    )
    seconds = time.time() - started
    return reply["choices"][0]["message"]["content"], seconds, reply["usage"]["completion_tokens"]


raw = []
seconds = []
generated = []
for n, row in enumerate(test, 1):
    text, took, tokens = ask(row["text"])
    raw.append(text)
    seconds.append(took)
    generated.append(tokens)
    if n % 10 == 0:
        print(n, round(sum(seconds) / n, 1), "s per receipt")

In [ ]:
everything = []
accepted = []
records = []

for row, text in zip(test, raw):
    gold = Receipt(**row["label"])
    pred = parse_receipt(text)
    problems = ["unparseable"] if pred is None else check(pred)
    result = score(pred or Receipt(), gold)
    everything.append(result)
    if not problems:
        accepted.append(result)
    records.append({"raw": text, "problems": problems})


def show(name, scores):
    summary = summarize(scores)
    cells = ", ".join(f"{key} {value:.1%}" for key, value in summary.items())
    print(f"{name} ({len(scores)}): {cells}")


print("unparseable:", sum(1 for r in records if r["problems"] == ["unparseable"]))
show("all answers", everything)
show("passed validators", accepted)
print("median seconds per receipt:", round(statistics.median(seconds), 1))
print("mean seconds per receipt:", round(statistics.mean(seconds), 1))
print("tokens per second:", round(sum(generated) / sum(seconds), 1))

with open(f"/kaggle/working/predictions_{QUANT}.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

## Publish the GGUF

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

if HF_REPO:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    api = HfApi(token=token)
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    api.upload_file(path_or_fileobj=gguf_path, path_in_repo=os.path.basename(gguf_path), repo_id=HF_REPO)
    print("uploaded", HF_REPO)